In [40]:
import fastf1
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

In [2]:
test = fastf1.get_session(2026, 'Brazil', 'Q')
test.load()

req         WARNING 	DEFAULT CACHE ENABLED! (918.01 MB) /Users/jakesherman/Library/Caches/fastf1
core           INFO 	Loading data for São Paulo Grand Prix - Qualifying [v3.9.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['1', '63', '12', '16', '30', '27', '6', '87', '14', '23', '44', '81', '51', '41', '55', '3', '43', '10', '18', '77', '11', '5'

In [3]:
test.laps

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 01:40:41.274000,NOR,1,NaT,1.0,1.0,0 days 00:14:18.887000,0 days 00:16:47.762000,NaT,NaT,...,True,McLaren,0 days 00:14:18.887000,2026-11-07 18:00:58.297,15,NaN,False,,False,False
1,0 days 01:43:14.243000,NOR,1,NaT,2.0,2.0,0 days 01:40:41.274000,NaT,NaT,0 days 00:01:01.924000,...,False,McLaren,0 days 01:40:41.274000,2026-11-07 19:27:20.684,12,NaN,False,,False,False
2,0 days 01:44:52.609000,NOR,1,0 days 00:01:38.366000,3.0,2.0,NaT,NaT,0 days 00:00:25.436000,0 days 00:00:51.379000,...,False,McLaren,0 days 01:43:14.243000,2026-11-07 19:29:53.653,1,NaN,False,,False,True
3,0 days 01:46:39.338000,NOR,1,0 days 00:01:46.729000,4.0,2.0,NaT,NaT,0 days 00:00:24.878000,0 days 00:00:57.169000,...,False,McLaren,0 days 01:44:52.609000,2026-11-07 19:31:32.019,1,NaN,False,,False,True
4,0 days 01:48:13.915000,NOR,1,0 days 00:01:34.577000,5.0,2.0,NaT,NaT,0 days 00:00:24.222000,0 days 00:00:49.531000,...,False,McLaren,0 days 01:46:39.338000,2026-11-07 19:33:18.748,12,NaN,False,,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
386,0 days 01:39:53.674000,BOR,5,NaT,1.0,1.0,0 days 00:15:14.825000,0 days 00:18:15.176000,NaT,NaT,...,True,Audi,0 days 00:15:14.825000,2026-11-07 18:01:54.235,15,NaN,False,,False,False
387,0 days 01:41:53.050000,BOR,5,NaT,2.0,2.0,0 days 01:39:53.674000,NaT,NaT,0 days 00:01:01.382000,...,True,Audi,0 days 01:39:53.674000,2026-11-07 19:26:33.084,1,NaN,False,,False,False
388,0 days 01:43:36.378000,BOR,5,0 days 00:01:43.328000,3.0,2.0,NaT,NaT,0 days 00:00:26.292000,0 days 00:00:54.853000,...,True,Audi,0 days 01:41:53.050000,2026-11-07 19:28:32.460,12,NaN,False,,False,True
389,0 days 01:45:17.942000,BOR,5,0 days 00:01:41.564000,4.0,2.0,NaT,NaT,0 days 00:00:26.239000,0 days 00:00:52.713000,...,True,Audi,0 days 01:43:36.378000,2026-11-07 19:30:15.788,1,NaN,False,,False,True


# Qualifying/Race Sessions (Brazil Only)

In [4]:
BRAZIL = {2018: 20, 2019: 20, 2021: 19, 2022: 21, 2023: 20, 2024: 21, 2025: 21, 2026: 20}
LAP_COLS = ["Driver", "Team", "LapNumber", "LapTime", "Sector1Time", "Sector2Time",
            "Sector3Time", "Compound", "TyreLife", "SpeedST", "Deleted",
            "IsAccurate", "PitOutTime", "PitInTime"]



In [5]:
def tag(df, session, year, rnd):
    df = df.copy()
    df["Year"] = year
    df["Round"] = rnd
    df["Session"] = session.name
    df['EventFormat'] = session.event['EventFormat']
    for col in df.select_dtypes('timedelta').columns:
        df[col] = df[col].dt.total_seconds()
    return df

In [6]:
q_laps, q_results, q_weather, r_results = [], [], [], []
for year, rnd in BRAZIL.items():
    quali = fastf1.get_session(year, rnd, 'Q')
    quali.load(telemetry=False)
    q_laps.append(tag(quali.laps[LAP_COLS], quali, year, rnd))
    q_results.append(tag(quali.results, quali, year, rnd,))
    q_weather.append(tag(quali.weather_data, quali, year, rnd))

    if year < 2026:
        race = fastf1.get_session(year, rnd, 'R')
        race.load(laps=False, weather=False, telemetry=False, messages=False)
        r_results.append(tag(race.results, race, year, rnd))

core           INFO 	Loading data for Brazilian Grand Prix - Qualifying [v3.9.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['44', '5', '77', '7', '33', '3', '9', '16', '8', '10', '20', '11', '31', '27', '35', '55', '28', '14', '18', '2']
core           INFO 	Loading data for Brazilian Grand Prix - Race [v3.9.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished lo

In [7]:
q_laps = pd.concat(q_laps, ignore_index=True)
q_results = pd.concat(q_results, ignore_index=True)
q_weather = pd.concat(q_weather, ignore_index=True)
r_results = pd.concat(r_results, ignore_index=True)


In [8]:
res = q_results.copy()
for seg in ['Q1', 'Q2', 'Q3']:
    fastest = res.groupby('Year')[seg].transform('min')
    res[f"{seg}_gap_pct"] = (res[seg] - fastest) / fastest * 100

res['FinalGapPct'] = res['Q3_gap_pct'].fillna(res['Q2_gap_pct']).fillna(res['Q1_gap_pct'])



In [9]:
res

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,Points,Laps,Year,Round,Session,EventFormat,Q1_gap_pct,Q2_gap_pct,Q3_gap_pct,FinalGapPct
0,44,L HAMILTON,HAM,hamilton,Mercedes,00D2BE,mercedes,Lewis,Hamilton,Lewis Hamilton,...,NaN,NaN,2018,20,Qualifying,conventional,0.379738,0.100403,0.000000,0.000000
1,5,S VETTEL,VET,vettel,Ferrari,DC0000,ferrari,Sebastian,Vettel,Sebastian Vettel,...,NaN,NaN,2018,20,Qualifying,conventional,0.362144,0.072349,0.138226,0.138226
2,77,V BOTTAS,BOT,bottas,Mercedes,00D2BE,mercedes,Valtteri,Bottas,Valtteri Bottas,...,NaN,NaN,2018,20,Qualifying,conventional,0.420790,0.000000,0.237809,0.237809
3,7,K RAIKKONEN,RAI,raikkonen,Ferrari,DC0000,ferrari,Kimi,Räikkönen,Kimi Räikkönen,...,NaN,NaN,2018,20,Qualifying,conventional,0.362144,0.444431,0.260103,0.260103
4,33,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,1E41FF,red_bull,Max,Verstappen,Max Verstappen,...,NaN,NaN,2018,20,Qualifying,conventional,0.000000,0.428190,0.738693,0.738693
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,10,P GASLY,GAS,gasly,Alpine,00A1E8,alpine,Pierre,Gasly,Pierre Gasly,...,NaN,NaN,2026,20,Qualifying,conventional,2.760092,NaN,NaN,2.760092
158,18,L STROLL,STR,stroll,Aston Martin,229971,aston_martin,Lance,Stroll,Lance Stroll,...,NaN,NaN,2026,20,Qualifying,conventional,2.984418,NaN,NaN,2.984418
159,77,V BOTTAS,BOT,bottas,Cadillac,909090,cadillac,Valtteri,Bottas,Valtteri Bottas,...,NaN,NaN,2026,20,Qualifying,conventional,3.117474,NaN,NaN,3.117474
160,11,S PEREZ,PER,perez,Cadillac,909090,cadillac,Sergio,Perez,Sergio Perez,...,NaN,NaN,2026,20,Qualifying,conventional,3.925708,NaN,NaN,3.925708


In [10]:
quali_feats = res[["Year", 
                   "Round", 
                   "EventFormat", 
                   "Abbreviation", 
                   "TeamName",
                   "Position", 
                   "Q1_gap_pct", 
                   "Q2_gap_pct", 
                   "Q3_gap_pct", 
                   "FinalGapPct"]]
quali_feats = quali_feats.rename(columns = {'Position' : 'QualiPos'})
print(quali_feats.shape)
print(quali_feats[quali_feats["QualiPos"] == 1])
print(quali_feats["FinalGapPct"].isna().sum())

(162, 10)
     Year  Round        EventFormat Abbreviation         TeamName  QualiPos  \
0    2018     20       conventional          HAM         Mercedes       1.0   
20   2019     20       conventional          VER  Red Bull Racing       1.0   
40   2021     19             sprint          HAM         Mercedes       1.0   
60   2022     21             sprint          MAG     Haas F1 Team       1.0   
80   2023     20    sprint_shootout          VER  Red Bull Racing       1.0   
100  2024     21  sprint_qualifying          NOR          McLaren       1.0   
120  2025     21  sprint_qualifying          NOR          McLaren       1.0   
140  2026     20       conventional          NOR          McLaren       1.0   

     Q1_gap_pct  Q2_gap_pct  Q3_gap_pct  FinalGapPct  
0      0.379738    0.100403         0.0          0.0  
20     0.000000    0.000000         0.0          0.0  
40     0.000000    0.000000         0.0          0.0  
60     1.159960    0.746321         0.0          0.0  
80 

In [11]:
quali_feats

,Year,Round,EventFormat,Abbreviation,TeamName,QualiPos,Q1_gap_pct,Q2_gap_pct,Q3_gap_pct,FinalGapPct
0,2018,20,conventional,HAM,Mercedes,1.0,0.379738,0.100403,0.000000,0.000000
1,2018,20,conventional,VET,Ferrari,2.0,0.362144,0.072349,0.138226,0.138226
2,2018,20,conventional,BOT,Mercedes,3.0,0.420790,0.000000,0.237809,0.237809
3,2018,20,conventional,RAI,Ferrari,4.0,0.362144,0.444431,0.260103,0.260103
4,2018,20,conventional,VER,Red Bull Racing,5.0,0.000000,0.428190,0.738693,0.738693
...,...,...,...,...,...,...,...,...,...,...
157,2026,20,conventional,GAS,Alpine,18.0,2.760092,NaN,NaN,2.760092
158,2026,20,conventional,STR,Aston Martin,19.0,2.984418,NaN,NaN,2.984418
159,2026,20,conventional,BOT,Cadillac,20.0,3.117474,NaN,NaN,3.117474
160,2026,20,conventional,PER,Cadillac,21.0,3.925708,NaN,NaN,3.925708


In [12]:
valid = q_laps[
    q_laps['LapTime'].notna()
    & (q_laps['Deleted']==False)
    & (q_laps['IsAccurate']==True)
    & q_laps['PitOutTime'].isna()
    & q_laps['PitInTime'].isna()

]

In [13]:
best_idx = valid.groupby(['Year', 'Driver'])['LapTime'].idxmin()
best = valid.loc[best_idx, ['Year', 'Driver', 'Compound', 'SpeedST']]
best = best.rename(columns={'Compound' : 'best_lap_compound', 
                           'SpeedST' : 
                           'best_lap_top_speed'})

In [14]:
counts = valid.groupby(['Year',
                        'Driver']).size().rename('n_flying_laps').reset_index()

In [15]:
best

,Year,Driver,best_lap_compound,best_lap_top_speed
280,2018,ALO,SUPERSOFT,316.0
46,2018,BOT,SUPERSOFT,330.0
117,2018,ERI,SUPERSOFT,325.0
170,2018,GAS,SUPERSOFT,324.0
151,2018,GRO,SUPERSOFT,321.0
...,...,...,...,...
2607,2026,PIA,INTERMEDIATE,228.0
2385,2026,RUS,INTERMEDIATE,254.0
2658,2026,SAI,WET,250.0
2704,2026,STR,WET,219.0


In [16]:
counts

,Year,Driver,n_flying_laps
0,2018,ALO,3
1,2018,BOT,7
2,2018,ERI,9
3,2018,GAS,11
4,2018,GRO,5
...,...,...,...
155,2026,PIA,13
156,2026,RUS,14
157,2026,SAI,11
158,2026,STR,5


In [17]:
lap_summary = counts.merge(best, on=['Year', 'Driver'])

In [18]:
lap_summary = lap_summary.rename(columns={'Driver': 'Abbreviation'})

In [19]:
lap_summary

,Year,Abbreviation,n_flying_laps,best_lap_compound,best_lap_top_speed
0,2018,ALO,3,SUPERSOFT,316.0
1,2018,BOT,7,SUPERSOFT,330.0
2,2018,ERI,9,SUPERSOFT,325.0
3,2018,GAS,11,SUPERSOFT,324.0
4,2018,GRO,5,SUPERSOFT,321.0
...,...,...,...,...,...
155,2026,PIA,13,INTERMEDIATE,228.0
156,2026,RUS,14,INTERMEDIATE,254.0
157,2026,SAI,11,WET,250.0
158,2026,STR,5,WET,219.0


In [20]:
quali_feats = quali_feats.merge(lap_summary, on=['Year', 'Abbreviation'], how='left')

In [21]:
quali_feats

,Year,Round,EventFormat,Abbreviation,TeamName,QualiPos,Q1_gap_pct,Q2_gap_pct,Q3_gap_pct,FinalGapPct,n_flying_laps,best_lap_compound,best_lap_top_speed
0,2018,20,conventional,HAM,Mercedes,1.0,0.379738,0.100403,0.000000,0.000000,7.0,SUPERSOFT,330.0
1,2018,20,conventional,VET,Ferrari,2.0,0.362144,0.072349,0.138226,0.138226,6.0,SUPERSOFT,329.0
2,2018,20,conventional,BOT,Mercedes,3.0,0.420790,0.000000,0.237809,0.237809,7.0,SUPERSOFT,330.0
3,2018,20,conventional,RAI,Ferrari,4.0,0.362144,0.444431,0.260103,0.260103,8.0,SUPERSOFT,326.0
4,2018,20,conventional,VER,Red Bull Racing,5.0,0.000000,0.428190,0.738693,0.738693,6.0,SUPERSOFT,321.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,2026,20,conventional,GAS,Alpine,18.0,2.760092,NaN,NaN,2.760092,9.0,WET,230.0
158,2026,20,conventional,STR,Aston Martin,19.0,2.984418,NaN,NaN,2.984418,5.0,WET,219.0
159,2026,20,conventional,BOT,Cadillac,20.0,3.117474,NaN,NaN,3.117474,9.0,WET,237.0
160,2026,20,conventional,PER,Cadillac,21.0,3.925708,NaN,NaN,3.925708,8.0,WET,226.0


In [22]:
q_weather

,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed,Year,Round,Session,EventFormat
0,58.555,23.5,72.4,924.2,False,37.9,145,2.3,2018,20,Qualifying,conventional
1,118.370,23.6,71.9,924.1,False,37.6,167,1.3,2018,20,Qualifying,conventional
2,178.386,23.4,72.4,924.1,False,37.8,125,1.7,2018,20,Qualifying,conventional
3,238.401,23.3,72.7,924.2,False,37.6,145,2.3,2018,20,Qualifying,conventional
4,298.416,23.3,72.9,924.2,False,37.5,163,1.5,2018,20,Qualifying,conventional
...,...,...,...,...,...,...,...,...,...,...,...,...
745,9348.395,20.7,89.3,924.3,True,36.6,24,2.1,2026,20,Qualifying,conventional
746,9408.392,20.7,89.3,924.3,True,36.6,24,2.1,2026,20,Qualifying,conventional
747,9468.396,20.7,90.3,924.5,True,29.7,22,1.4,2026,20,Qualifying,conventional
748,9528.400,20.6,89.9,924.8,True,28.0,336,1.1,2026,20,Qualifying,conventional


In [23]:
weather_summary = q_weather.groupby('Year').agg(
    air_temp=('AirTemp', 'mean'),
    track_temp=('TrackTemp', 'mean' ),
    humidity=('Humidity', 'mean'),
    wind_speed=('WindSpeed', 'mean'),
    any_rain=('Rainfall', 'max')
).reset_index()


In [24]:
weather_summary['any_rain'] = weather_summary['any_rain'].astype(int)

In [25]:
weather_summary

,Year,air_temp,track_temp,humidity,wind_speed,any_rain
0,2018,22.980769,35.606410,74.773077,1.629487,1
1,2019,20.682051,33.889744,66.548718,1.389744,0
2,2021,15.702564,24.988462,80.185897,0.464103,0
3,2022,19.164198,26.713580,93.580247,1.135802,1
4,2023,29.379070,38.333721,57.337209,2.409302,1
5,2024,20.271963,23.379439,93.355140,0.580374,1
6,2025,25.970732,40.280488,56.451220,1.541463,0
7,2026,20.547500,30.590000,91.885625,1.363750,1


In [26]:
quali_feats = quali_feats.merge(weather_summary, on='Year', how='left')

In [27]:
quali_feats

,Year,Round,EventFormat,Abbreviation,TeamName,QualiPos,Q1_gap_pct,Q2_gap_pct,Q3_gap_pct,FinalGapPct,n_flying_laps,best_lap_compound,best_lap_top_speed,air_temp,track_temp,humidity,wind_speed,any_rain
0,2018,20,conventional,HAM,Mercedes,1.0,0.379738,0.100403,0.000000,0.000000,7.0,SUPERSOFT,330.0,22.980769,35.60641,74.773077,1.629487,1
1,2018,20,conventional,VET,Ferrari,2.0,0.362144,0.072349,0.138226,0.138226,6.0,SUPERSOFT,329.0,22.980769,35.60641,74.773077,1.629487,1
2,2018,20,conventional,BOT,Mercedes,3.0,0.420790,0.000000,0.237809,0.237809,7.0,SUPERSOFT,330.0,22.980769,35.60641,74.773077,1.629487,1
3,2018,20,conventional,RAI,Ferrari,4.0,0.362144,0.444431,0.260103,0.260103,8.0,SUPERSOFT,326.0,22.980769,35.60641,74.773077,1.629487,1
4,2018,20,conventional,VER,Red Bull Racing,5.0,0.000000,0.428190,0.738693,0.738693,6.0,SUPERSOFT,321.0,22.980769,35.60641,74.773077,1.629487,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,2026,20,conventional,GAS,Alpine,18.0,2.760092,NaN,NaN,2.760092,9.0,WET,230.0,20.547500,30.59000,91.885625,1.363750,1
158,2026,20,conventional,STR,Aston Martin,19.0,2.984418,NaN,NaN,2.984418,5.0,WET,219.0,20.547500,30.59000,91.885625,1.363750,1
159,2026,20,conventional,BOT,Cadillac,20.0,3.117474,NaN,NaN,3.117474,9.0,WET,237.0,20.547500,30.59000,91.885625,1.363750,1
160,2026,20,conventional,PER,Cadillac,21.0,3.925708,NaN,NaN,3.925708,8.0,WET,226.0,20.547500,30.59000,91.885625,1.363750,1


In [28]:
r_results

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,Q2,Q3,Time,Status,Points,Laps,Year,Round,Session,EventFormat
0,44,L HAMILTON,HAM,hamilton,Mercedes,00D2BE,mercedes,Lewis,Hamilton,Lewis Hamilton,...,NaN,NaN,5229.066,Finished,25.0,71.0,2018,20,Race,conventional
1,33,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,1E41FF,red_bull,Max,Verstappen,Max Verstappen,...,NaN,NaN,1.469,Finished,18.0,71.0,2018,20,Race,conventional
2,7,K RAIKKONEN,RAI,raikkonen,Ferrari,DC0000,ferrari,Kimi,Räikkönen,Kimi Räikkönen,...,NaN,NaN,4.764,Finished,15.0,71.0,2018,20,Race,conventional
3,3,D RICCIARDO,RIC,ricciardo,Red Bull Racing,1E41FF,red_bull,Daniel,Ricciardo,Daniel Ricciardo,...,NaN,NaN,5.193,Finished,12.0,71.0,2018,20,Race,conventional
4,77,V BOTTAS,BOT,bottas,Mercedes,00D2BE,mercedes,Valtteri,Bottas,Valtteri Bottas,...,NaN,NaN,22.943,Finished,10.0,71.0,2018,20,Race,conventional
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,18,L STROLL,STR,stroll,Aston Martin,229971,aston_martin,Lance,Stroll,Lance Stroll,...,NaN,NaN,58.247,Finished,0.0,71.0,2025,21,Race,sprint_qualifying
136,22,Y TSUNODA,TSU,tsunoda,Red Bull Racing,4781D7,red_bull,Yuki,Tsunoda,Yuki Tsunoda,...,NaN,NaN,69.176,Finished,0.0,71.0,2025,21,Race,sprint_qualifying
137,44,L HAMILTON,HAM,hamilton,Ferrari,ED1131,ferrari,Lewis,Hamilton,Lewis Hamilton,...,NaN,NaN,NaN,Retired,0.0,37.0,2025,21,Race,sprint_qualifying
138,16,C LECLERC,LEC,leclerc,Ferrari,ED1131,ferrari,Charles,Leclerc,Charles Leclerc,...,NaN,NaN,NaN,Retired,0.0,5.0,2025,21,Race,sprint_qualifying


In [29]:
race_cols = r_results[['Year', 
                       'Abbreviation', 
                       'Position', 
                       'GridPosition', 
                       'Points', 
                       'Status']]
race_cols = race_cols.rename(columns={'Position': 'FinishPos'})
race_cols['Podium'] = (race_cols['FinishPos'] <= 3).astype(int)

In [30]:
race_cols

,Year,Abbreviation,FinishPos,GridPosition,Points,Status,Podium
0,2018,HAM,1.0,1.0,25.0,Finished,1
1,2018,VER,2.0,5.0,18.0,Finished,1
2,2018,RAI,3.0,4.0,15.0,Finished,1
3,2018,RIC,4.0,11.0,12.0,Finished,0
4,2018,BOT,5.0,3.0,10.0,Finished,0
...,...,...,...,...,...,...,...
135,2025,STR,16.0,14.0,0.0,Finished,0
136,2025,TSU,17.0,17.0,0.0,Finished,0
137,2025,HAM,18.0,13.0,0.0,Retired,0
138,2025,LEC,19.0,3.0,0.0,Retired,0


In [31]:
quali_feats = quali_feats.merge(race_cols, on=['Year', 'Abbreviation'], how='left')

In [32]:
quali_feats

,Year,Round,EventFormat,Abbreviation,TeamName,QualiPos,Q1_gap_pct,Q2_gap_pct,Q3_gap_pct,FinalGapPct,...,air_temp,track_temp,humidity,wind_speed,any_rain,FinishPos,GridPosition,Points,Status,Podium
0,2018,20,conventional,HAM,Mercedes,1.0,0.379738,0.100403,0.000000,0.000000,...,22.980769,35.60641,74.773077,1.629487,1,1.0,1.0,25.0,Finished,1.0
1,2018,20,conventional,VET,Ferrari,2.0,0.362144,0.072349,0.138226,0.138226,...,22.980769,35.60641,74.773077,1.629487,1,6.0,2.0,8.0,Finished,0.0
2,2018,20,conventional,BOT,Mercedes,3.0,0.420790,0.000000,0.237809,0.237809,...,22.980769,35.60641,74.773077,1.629487,1,5.0,3.0,10.0,Finished,0.0
3,2018,20,conventional,RAI,Ferrari,4.0,0.362144,0.444431,0.260103,0.260103,...,22.980769,35.60641,74.773077,1.629487,1,3.0,4.0,15.0,Finished,1.0
4,2018,20,conventional,VER,Red Bull Racing,5.0,0.000000,0.428190,0.738693,0.738693,...,22.980769,35.60641,74.773077,1.629487,1,2.0,5.0,18.0,Finished,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,2026,20,conventional,GAS,Alpine,18.0,2.760092,NaN,NaN,2.760092,...,20.547500,30.59000,91.885625,1.363750,1,NaN,NaN,NaN,NaN,NaN
158,2026,20,conventional,STR,Aston Martin,19.0,2.984418,NaN,NaN,2.984418,...,20.547500,30.59000,91.885625,1.363750,1,NaN,NaN,NaN,NaN,NaN
159,2026,20,conventional,BOT,Cadillac,20.0,3.117474,NaN,NaN,3.117474,...,20.547500,30.59000,91.885625,1.363750,1,NaN,NaN,NaN,NaN,NaN
160,2026,20,conventional,PER,Cadillac,21.0,3.925708,NaN,NaN,3.925708,...,20.547500,30.59000,91.885625,1.363750,1,NaN,NaN,NaN,NaN,NaN


In [33]:
quali_feats.loc[quali_feats['GridPosition']==0, 'GridPosition']=21

In [34]:
quali_feats['is_sprint'] = (quali_feats['EventFormat'] != 'conventional').astype(int)

In [35]:
quali_feats['best_lap_wet'] = quali_feats['best_lap_compound'].isin(['INTERMEDIATE', 'WET']).astype(int)

In [36]:
FEATURES = ['QualiPos', 'Q1_gap_pct', 'Q2_gap_pct', 'Q3_gap_pct', 'FinalGapPct',
            'n_flying_laps', 'best_lap_top_speed', 'best_lap_wet', 'is_sprint']


# Building Model

In [37]:
train = quali_feats[quali_feats['Year'] < 2026]
test = quali_feats[quali_feats['Year'] == 2026]

In [38]:
X_train = train[FEATURES]
y_train = train['Podium'].astype(int)
X_test = test[FEATURES]
print(X_train.shape, y_train.shape, X_test.shape)

(140, 9) (140,) (22, 9)


In [39]:
hits = {}
for year, grp in train.groupby('Year'):
    hits[year] = int(grp.nsmallest(3, 'QualiPos')['Podium'].sum())
print(hits, sum(hits.values()), 'of', 3 * len(hits))

{2018: 1, 2019: 1, 2021: 3, 2022: 1, 2023: 1, 2024: 0, 2025: 2} 9 of 21


In [41]:
final_model = HistGradientBoostingClassifier(
    max_depth=3, learning_rate=0.05, max_iter=100, 
    min_samples_leaf=10, random_state=0
)
final_model.fit(X_train, y_train)

,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.05
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",3
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",10
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",0
,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""Categorical"" and ""Enum"" are considered to be categorical features. The input must be a dataframe that is supported by narwhals (or supports it): :func:`narwhals.from_native` must work. This is the case, for instance, for pandas and polars DataFrames.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values

In [44]:
pred = test[['Abbreviation', 'TeamName', 'QualiPos']].copy()
pred['podium_prob'] = final_model.predict_proba(X_test)[:,1]
pred = pred.sort_values('podium_prob', ascending=False)

In [48]:
print(pred.head(8))
print(pred['podium_prob'].sum())

    Abbreviation         TeamName  QualiPos  podium_prob
141          RUS         Mercedes       2.0     0.853036
140          NOR          McLaren       1.0     0.731830
145          HUL             Audi       6.0     0.581668
144          LAW     Racing Bulls       5.0     0.550068
155          VER  Red Bull Racing      16.0     0.331578
142          ANT         Mercedes       3.0     0.223727
146          HAD  Red Bull Racing       7.0     0.148398
150          HAM          Ferrari      11.0     0.117863
4.01310858438588


In [49]:
pred

,Abbreviation,TeamName,QualiPos,podium_prob
141,RUS,Mercedes,2.0,0.853036
140,NOR,McLaren,1.0,0.731830
145,HUL,Audi,6.0,0.581668
144,LAW,Racing Bulls,5.0,0.550068
155,VER,Red Bull Racing,16.0,0.331578
142,ANT,Mercedes,3.0,0.223727
146,HAD,Red Bull Racing,7.0,0.148398
150,HAM,Ferrari,11.0,0.117863
143,LEC,Ferrari,4.0,0.090643
148,ALO,Aston Martin,9.0,0.089020


In [50]:
from sklearn.ensemble import HistGradientBoostingRegressor

In [51]:
final_model2 = HistGradientBoostingRegressor(
    max_depth=3, learning_rate=0.05, max_iter=100,
    min_samples_leaf=10, random_state=0
)
final_model2.fit(X_train, y_train)


,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.05
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",3
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",10
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",0
,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255


In [55]:
pred2 = test[['Abbreviation', 'TeamName', 'QualiPos']]
pred2['podium_prob'] = final_model.predict_proba(X_test)[:,1]
pred2 = pred2.sort_values('podium_prob', ascending=False)

/var/folders/12/drpcbrkj1g9gsv0sh2snq2fw0000gn/T/ipykernel_68535/1354262837.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pred2['podium_prob'] = final_model.predict_proba(X_test)[:,1]


In [56]:
pred2

,Abbreviation,TeamName,QualiPos,podium_prob
141,RUS,Mercedes,2.0,0.853036
140,NOR,McLaren,1.0,0.731830
145,HUL,Audi,6.0,0.581668
144,LAW,Racing Bulls,5.0,0.550068
155,VER,Red Bull Racing,16.0,0.331578
142,ANT,Mercedes,3.0,0.223727
146,HAD,Red Bull Racing,7.0,0.148398
150,HAM,Ferrari,11.0,0.117863
143,LEC,Ferrari,4.0,0.090643
148,ALO,Aston Martin,9.0,0.089020
